# Combined Matrix Pipeline

Builds `combined_matrix.csv` (and `combined_results_with_features.csv`) from the MATLAB session file and `Cluster_detail_results.csv`.

User-configurable section:
- `FOLDER_TO_WEEK`: dictionary mapping a `Folder_Name` (or substring) to the stage / week number. This is the global constant `get_week_number()` will consult when assigning a `Week_Number` to each row.

**Installations**

Uncomment and run the first time you use this notebook.

In [ ]:
# ! pip install h5py pandas numpy

In [ ]:
# Imports
import os
import re
import h5py
import numpy as np
import pandas as pd

**File directories**

Make changes for the input and output paths here.

In [ ]:
# Input files used for analysis
Cluster_detail_results = pd.read_csv(r'Y:\Members\Mia-Sanjana-Hadent\Processed Data\wildtype_062425_011725_9\Altogether Clustering\Cluster_detail_results.csv')
mat_filename = r"Y:\Members\Mia-Sanjana-Hadent\Processed Data\wildtype_062425_011725_9\Altogether Clustering\session_1_out.mat"
filename_groups = r'Y:\Members\Mia-Sanjana-Hadent\Processed Data\wildtype_062425_011725_9\Altogether Clustering\cluster_groups_matlab.csv'

# Output directory for the combined matrix CSV
output_path = r"Y:\Members\Mia-Sanjana-Hadent\Visualizations\Output_wildtype_test - testing entropy\data"
os.makedirs(output_path, exist_ok=True)

# Use any label for mouse
mouse_name = "wildtype"

**Folder name to Week number mapping**

`FOLDER_TO_WEEK` is the **global constant** that tells `get_week_number()` what `Week_Number` (stage) each folder name corresponds to.

- Keys are matched against the `Folder_Name` column in `Cluster_detail_results.csv` using a case-insensitive substring match.
- Values are the stage / week number (1-indexed) for that folder.
- Add one entry per folder; you can have as many stages as you need.

If a `Folder_Name` does not match any key, the legacy fallback patterns (`weekXX`, `restricted`, `open`, `arenal/m/h`) are checked. Anything still unmatched returns `None` and is forward-filled.

In [ ]:
# -------------------- USER INPUT --------------------
# Define one entry per folder. The folder name can be a substring of the
# actual Folder_Name in Cluster_detail_results.csv (case-insensitive).
FOLDER_TO_WEEK = {
    # "folder_name_for_stage_1": 1,
    # "folder_name_for_stage_2": 2,
    # "folder_name_for_stage_3": 3,
    # "folder_name_for_stage_4": 4,
    # "folder_name_for_stage_5": 5,
}

# Set to False to disable the legacy weekXX / arena fallback below
USE_LEGACY_FALLBACK = True
# ----------------------------------------------------


def get_week_number(folder_name):
    """Map a Folder_Name string to a Week_Number (stage).

    Resolution order:
      1. FOLDER_TO_WEEK (user-defined global constant; substring, case-insensitive).
      2. Legacy regex/keyword fallback (only if USE_LEGACY_FALLBACK is True).
      3. None.
    """
    if not isinstance(folder_name, str):
        return None

    folder_name_lower = folder_name.lower()

    # 1. User-defined mapping
    for key, week in FOLDER_TO_WEEK.items():
        if key and key.lower() in folder_name_lower:
            return int(week)

    if not USE_LEGACY_FALLBACK:
        return None

    # 2. Legacy fallback - weekXX pattern
    match = re.search(r'week(\d+)', folder_name_lower)
    if match:
        return int(match.group(1))

    # 2. Legacy fallback - arena names
    if "restricted" in folder_name_lower:
        return 1
    if "open" in folder_name_lower:
        return 2
    if "arenal" in folder_name_lower:
        return 3
    if "arenam" in folder_name_lower:
        return 4
    if "arenah" in folder_name_lower:
        return 5

    return None

**Feature extraction helpers**

In [ ]:
# --------------------------- Extracting 4 Features ---------------------------
def find_path_to_key(h5_obj, target_key, path=""):
    for key in h5_obj.keys():
        new_path = f"{path}/{key}"
        if key == target_key:
            return new_path
        if isinstance(h5_obj[key], h5py.Group):
            result = find_path_to_key(h5_obj[key], target_key, new_path)
            if result is not None:
                return result
    return None


def load_funct_features(filepath):
    with h5py.File(filepath, "r") as f:
        struct_path = find_path_to_key(f, "StructData")
        if struct_path is None:
            raise KeyError("Could not find 'StructData' in MAT file.")

        struct_group = f[struct_path]
        if "func" not in struct_group:
            raise KeyError("'func' not found inside StructData.")

        func = struct_group["func"]
        # MATLAB {1,1-4} -> Python func[0-3][0]
        feature_refs = [func[i][0] for i in range(4)]
        feature_arrays = []
        for ref in feature_refs:
            arr = f[ref][()]
            arr = np.array(arr).squeeze()
            feature_arrays.append(arr)

        return np.vstack(feature_arrays)


def combine_results(mat_file, cb_matrix, output_file="combined_results_with_features.csv"):
    features = load_funct_features(mat_file)

    n_samples = features.shape[1]
    if n_samples % 60 != 0:
        raise ValueError("Number of samples is not divisible by 60.")

    log_tot_accel = features[3]
    exp_log_tot_accel = np.exp(log_tot_accel)
    averaged_tot_accel = exp_log_tot_accel.reshape(-1, 60).mean(axis=1)

    features_reshaped = features.reshape(4, -1, 60)
    features_avg = features_reshaped.mean(axis=2)

    df = cb_matrix.copy()
    if len(df) != features_avg.shape[1]:
        raise ValueError(
            f"CSV rows ({len(df)}) do not match averaged func length ({features_avg.shape[1]})."
        )

    df["anterior_posterior_x_accel"] = features_avg[0]
    df["dorsal_ventral_y_accel"] = features_avg[1]
    df["y_gyro"] = features_avg[2]
    df["TotAccelBA"] = averaged_tot_accel

    df.to_csv(output_file, index=False)
    print("Saved:", output_file)
    return df

**Build the combined matrix**

In [ ]:
grouped_clusters = pd.read_csv(filename_groups)

# Read in values from clustersIdx, set automatically to the cluster value it represents rather than the actual value of the cell
if os.path.exists(mat_filename):
    with h5py.File(mat_filename, 'r') as file:
        library = file['Library']
        clustersIdx = library['clustersIdx']

        group_labels = []
        for i in range(clustersIdx.shape[0]):
            idxref = clustersIdx[i, 0]
            cluster_indices = np.array(file[idxref]).flatten()
            group_labels.extend([i + 1] * len(cluster_indices))

        group_labels_df = pd.DataFrame(group_labels, columns=['GroupLabel'])
        print(f"Group labels shape: {group_labels_df.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')


# Read in the feature values from clusters and stack on top of each other
if os.path.exists(mat_filename):
    print('Loading session file from expected Results/test1 directory')
    with h5py.File(mat_filename, 'r') as file:
        if 'Library' not in file:
            raise KeyError("The key 'Library' was not found in the session file.")
        library = file['Library']
        clusters = library['clusters']

        features_data = []
        for i in range(clusters.shape[0]):
            cref = clusters[i, 0]
            feature_data = np.array(file[cref]).T
            features_data.append(feature_data)

        features_matrix = np.vstack(features_data)
        features_df = pd.DataFrame(features_matrix)
        print(f"Loaded {len(features_data)} clusters with total points: {features_matrix.shape[0]}")
        print(f"Combined matrix shape: {features_matrix.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')

features_df['cluster'] = group_labels_df['GroupLabel'].values

In [ ]:
# Create combined matrix by matching first occurrence of each cluster in features_df to
# first occurrence in Cluster_detail_results, keeping order of Cluster_detail_results
def match_features_to_details(features_df, detail_df):
    matched_rows = []
    cluster_counts = {}
    for _, detail_row in detail_df.iterrows():
        cluster_val = detail_row['ClusterIdx']
        count = cluster_counts.get(cluster_val, 0)
        feature_rows = features_df[features_df['cluster'] == cluster_val]
        if count < len(feature_rows):
            feature_row = feature_rows.iloc[count]
            combined_row = feature_row.to_dict()
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            combined_row['Folder_Name'] = detail_row['Folder_Name'] if 'Folder_Name' in detail_row else None
            matched_rows.append(combined_row)
            cluster_counts[cluster_val] = count + 1
        else:
            combined_row = {col: None for col in features_df.columns}
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            combined_row['Folder_Name'] = detail_row['Folder_Name'] if 'Folder_Name' in detail_row else None
            matched_rows.append(combined_row)
    return pd.DataFrame(matched_rows)


combined_matrix = match_features_to_details(features_df, Cluster_detail_results)

# Apply user-controlled Week_Number assignment via get_week_number
combined_matrix['Week_Number'] = combined_matrix['Folder_Name'].apply(get_week_number)

# Forward-fill any rows whose Folder_Name was not mappable
combined_matrix['Week_Number'] = combined_matrix['Week_Number'].ffill()

unmapped = combined_matrix['Week_Number'].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows still have no Week_Number after forward-fill. "
          f"Check FOLDER_TO_WEEK against the unique Folder_Name values below:")
    print(combined_matrix.loc[combined_matrix['Week_Number'].isna(), 'Folder_Name'].unique())
else:
    combined_matrix['Week_Number'] = combined_matrix['Week_Number'].astype(int)

print("Detected Week_Numbers:", sorted(combined_matrix['Week_Number'].dropna().unique()))

In [ ]:
# Clean up column names: Feature1..Feature30, Timestamp, Cluster, Week_Number, Group
feature_cols = {i: f'Feature{i+1}' for i in range(30)}
combined_matrix = combined_matrix.rename(columns=feature_cols)
combined_matrix = combined_matrix.drop(columns=['cluster'])
combined_matrix = combined_matrix.drop(columns=['Folder_Name'])
combined_matrix = combined_matrix.rename(columns={'ClusterIdx': 'Cluster'})

combined_matrix["Group"] = combined_matrix["Cluster"].map(
    grouped_clusters.set_index("Cluster")["Group"]
)

# Save combined_matrix and combined_results_with_features files
output_file = os.path.join(output_path, f"combined_matrix_{mouse_name}.csv")
combined_matrix.to_csv(output_file, index=False)
print(f"Saved as {output_file}")

combined_matrix = combine_results(
    mat_filename,
    combined_matrix,
    output_file=os.path.join(output_path, f"combined_results_{mouse_name}_with_features.csv")
)
print(combined_matrix.head())